# featherweight-ai — Kaggle QLoRA training run (Week 3)

Produces **benchmark row 2**. A thin launcher, not a place logic lives
(`docs/plan.md` §4): every cell below either configures the session or calls into
the repo.

**Before running — right sidebar:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Add-ons → Secrets | `HF_TOKEN` = your Hugging Face read token |

The model is gated (`gated: auto`) — a token alone is not enough, the licence must
also have been accepted on the model page. Those are two separate things and the
failure mode is a 403 that looks like a bad token (`memory.md` §8).

**Budget:** one run is **60 minutes of wall-clock training**, not a fixed number of
steps. `plan.md` §6 compares rows at equal wall-clock and equal peak VRAM — DoRA
costs more per step than LoRA, so matching on steps would quietly hand the slower
method more compute. Everything after the training cell is minutes, not hours.

## 1. What hardware did we actually get?

Record it. A wall-clock budget is only comparable across runs on the same device.

In [3]:
!nvidia-smi

Sat Aug 15 10:45:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo

Code lives in git; the notebook is a launcher. `rm -rf` first so a re-run inside the
same session picks up a fresh push rather than silently testing stale code.

In [4]:
!rm -rf /kaggle/working/featherweight-ai
!git clone -q https://github.com/AadiPathak23/featherweight-ai.git /kaggle/working/featherweight-ai
%cd /kaggle/working/featherweight-ai
!git log --oneline -1

/kaggle/working/featherweight-ai
cbe8125 (HEAD -> main, origin/main, origin/HEAD) Kaggle: remove torchao after install — peft raises on the image's 0.10.0


## 3. Install — the one genuinely untested piece

`requirements-kaggle.txt` had never run against this image before 2026-08-15. It
deliberately does **not** list `torch`: Kaggle's is preinstalled and matched to
CUDA 12.8, and letting pip resolve its own would mean a ~2.5 GB download and a real
chance of a CUDA mismatch that only fails at the first `.cuda()` call.

🚨 **What it actually found.** Not torch — `torchao`. The image ships 0.10.0, and
peft's availability check *raises* on an old torchao instead of returning `False`,
killing LoRA injection. It reproduces on both peft 0.19.1 and 0.20.0, and it cannot
happen locally because torchao is not installed there at all. **The untested piece
failed in the one dimension a Windows laptop could not have predicted: not a version
this file pins, but a package the base image already had.**

So the next cell **prints every resolved version**. If this run is ever compared
against a later one, the difference has to be visible here rather than inferred.

In [5]:
!pip install -q -r requirements-kaggle.txt

# Kaggle's base image ships torchao 0.10.0, and peft's is_torchao_available()
# RAISES ImportError when torchao is present but < 0.16.0 -- it does not return
# False. LoRA injection calls it on every target module, so get_peft_model() dies
# before a single step. Measured 2026-08-15 on peft 0.19.1; peft 0.20.0 carries the
# identical raise, so pinning peft is NOT the fix.
#
# We never use torchao -- bitsandbytes does the quantization -- and with the package
# absent the check short-circuits on find_spec() and returns False cleanly. Removing
# an unused dependency is the targeted fix; upgrading torchao would add a
# torch-version-coupled package we have no use for.
!pip uninstall -y -q torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 50.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.8 MB/s eta 0:00:00:00:0100:01


In [6]:
import torch, transformers, peft, bitsandbytes, accelerate, datasets, huggingface_hub, PIL, sys

print("python           :", sys.version.split()[0])
print("torch            :", torch.__version__, "| CUDA:", torch.version.cuda)
print("transformers     :", transformers.__version__, " (must be >=4.57, <5)")
print("peft             :", peft.__version__)
print("bitsandbytes     :", bitsandbytes.__version__)
print("accelerate       :", accelerate.__version__)
print("datasets         :", datasets.__version__)
print("huggingface_hub  :", huggingface_hub.__version__, " (must be <1.0)")
print("pillow           :", PIL.__version__)

print("\n--- devices ---")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, capability=({p.major}, {p.minor}), {p.total_memory/1024**3:.2f} GiB")

# The constraint the whole project is built around. Bare is_bf16_supported() returns
# True on a T4 because PyTorch emulates bf16 through fp32 — functional, slow, none of
# the benefit. `including_emulation=False` is the truth. memory.md §5.
print("\nbf16 (bare, LIES on a T4) :", torch.cuda.is_bf16_supported())
try:
    print("bf16 (native, the truth)  :", torch.cuda.is_bf16_supported(including_emulation=False))
except TypeError:
    print("bf16 (native, the truth)  : <arg not in this torch>", torch.cuda.get_device_capability(0) >= (8, 0))

python           : 3.12.13
torch            : 2.10.0+cu128 | CUDA: 12.8
transformers     : 4.57.6  (must be >=4.57, <5)
peft             : 0.19.1
bitsandbytes     : 0.50.1
accelerate       : 1.13.0
datasets         : 5.0.0
huggingface_hub  : 0.36.2  (must be <1.0)
pillow           : 11.3.0

--- devices ---
  cuda:0 = Tesla T4, capability=(7, 5), 14.56 GiB
  cuda:1 = Tesla T4, capability=(7, 5), 14.56 GiB

bf16 (bare, LIES on a T4) : True
bf16 (native, the truth)  : False


## 4. HF token from Secrets

Never typed into a cell. Kaggle notebooks are public by default and a committed
token is a leaked token — that has already happened once in this project
(`memory.md` §1 security note). Prints the length only.

In [7]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN retrieved OK, length:", len(os.environ["HF_TOKEN"]))

HF_TOKEN retrieved OK, length: 37


## 5. Build the data

Two artifacts. Both are rebuilt from **committed manifests**, so the identity is
fixed in git and only the pixels are regenerated here.

- **eval split** — `night-validation` shards 0–4: **659 rows over 115 images**
- **training pool** — the **whole day domain**, 13 shards: **1,817 rows over 276 images**

`build_train_pool.py` runs a **hard image-leak check** between the two and exits
non-zero if a single image appears in both.

🚨 **This check has already fired once, and it is why the eval split is night.** It
was written expecting to print `0`. It printed **235**: of the 241 images in
`day-train` shards 0–3, 235 also sat in the then-frozen day eval split, *byte-identical
by sha256* — while **0 of 560 `(image, question, answer)` triples** were shared. The
dataset's own `day-train` / `day-validation` labels are two sets of **questions about
the same ~276 keyframes**, not two sets of images. The answers never leaked; the pixels
almost entirely did. Decision **D11** in `memory.md`, **Amendment 1** in
`eval-protocol.md`.

Day and night are different drives, so `day ∩ night = 0` images — measured in both
directions, not assumed, because this dataset's split names have already been shown
not to mean what they say.

⏱️ **Cost on Kaggle:** the HF cache starts empty, so this pulls ~18 shards at ~460 MB
each (**≈8 GB**) before decoding. Locally the pool decodes at 74 images/min. Budget
the time; it is the longest cell before training.

In [8]:
!python scripts/build_eval_split.py --split night


--- eval split — night (5 shards) -------------------------------
night-validation/data-00000-of-00005.arr(…): 100%|█| 452M/452M [00:02<00:00, 202
  night-validation/data-00000-of-00005.arrow  (431 MB)
    night-validation/data-00000-of-00005.arrow  ->  132 rows (132 newly encoded)
night-validation/data-00001-of-00005.arr(…): 100%|█| 452M/452M [00:02<00:00, 204
  night-validation/data-00001-of-00005.arrow  (431 MB)
    night-validation/data-00001-of-00005.arrow  ->  132 rows (132 newly encoded)
night-validation/data-00002-of-00005.arr(…): 100%|█| 452M/452M [00:02<00:00, 204
  night-validation/data-00002-of-00005.arrow  (431 MB)
    night-validation/data-00002-of-00005.arrow  ->  132 rows (132 newly encoded)
night-validation/data-00003-of-00005.arr(…): 100%|█| 452M/452M [00:02<00:00, 204
  night-validation/data-00003-of-00005.arrow  (431 MB)
    night-validation/data-00003-of-00005.arrow  ->  132 rows (132 newly encoded)
night-validation/data-00004-of-00005.arr(…): 100%|█| 449M/449M [0

In [9]:
!python scripts/build_train_pool.py


--- training pool (the day domain — 13 shards) ------------------
day-train/data-00000-of-00016.arrow: 100%|████| 479M/479M [00:02<00:00, 198MB/s]
  day-train/data-00000-of-00016.arrow  (457 MB)
    day-train/data-00000-of-00016.arrow  ->  140 rows (140 newly encoded)
day-train/data-00001-of-00016.arrow: 100%|███| 479M/479M [00:06<00:00, 79.7MB/s]
  day-train/data-00001-of-00016.arrow  (457 MB)
    day-train/data-00001-of-00016.arrow  ->  140 rows (140 newly encoded)
day-train/data-00002-of-00016.arrow: 100%|███| 479M/479M [00:12<00:00, 38.6MB/s]
  day-train/data-00002-of-00016.arrow  (457 MB)
    day-train/data-00002-of-00016.arrow  ->  140 rows (140 newly encoded)
day-train/data-00003-of-00016.arrow: 100%|████| 479M/479M [00:03<00:00, 149MB/s]
  day-train/data-00003-of-00016.arrow  (457 MB)
    day-train/data-00003-of-00016.arrow  ->  140 rows (140 newly encoded)
day-train/data-00004-of-00016.arrow: 100%|████| 479M/479M [00:02<00:00, 184MB/s]
  day-train/data-00004-of-00016.arrow  (

## 6. Measure the VRAM/batch-size curve *before* committing an hour to it

One point cannot separate the **fixed** cost (4-bit weights, the fp32 upcast of
`embed_tokens`, optimizer state) from the part that **scales with batch size**
(retained activations). Two can.

⚠️ **Write prediction #2 in `docs/learning-log.md` before running this cell.** It has
been deliberately left unrun so it stays a real prediction — the mechanism (which parts
of the measured 3.60 GiB scale and which do not) is the answer, not the number.

This run does **not** consume the result: the training cell below is pinned to batch
size 1 to match the 3060 row. The curve is measured here for **Week 4**, where all
three methods need a batch-size policy chosen on the device they actually run on.

In [10]:
!python -m src.train --probe-batch --probe-max 32 --out train_batch_probe_t4.json


--- QLoRA training — Week 3 -------------------------------------
Device  : Tesla T4
Budget  : 60 min wall-clock
Optim   : lr=0.0001 warmup=20 schedule=constant clip=None
Batch   : 1 physical x 8 accum = 8 effective
LoRA    : r=8 alpha=16 dropout=0.05

--- training pool (day-train — never the eval split) ------------
  1817 rows, 276 distinct scenes

--- model -------------------------------------------------------
preprocessor_config.json: 100%|████████████████| 390/390 [00:00<00:00, 3.99MB/s]
tokenizer_config.json: 10.9kB [00:00, 33.8MB/s]
vocab.json: 2.78MB [00:00, 40.0MB/s]
merges.txt: 1.67MB [00:00, 92.8MB/s]
tokenizer.json: 7.03MB [00:00, 27.6MB/s]
video_preprocessor_config.json: 100%|██████████| 385/385 [00:00<00:00, 3.86MB/s]
chat_template.json: 5.50kB [00:00, 25.7MB/s]
config.json: 1.50kB [00:00, 7.94MB/s]
model.safetensors: 100%|████████████████████| 4.88G/4.88G [00:32<00:00, 151MB/s]
generation_config.json: 100%|██████████████████| 269/269 [00:00<00:00, 2.28MB/s]
  [VRAM] a

## 7. Train

`src/train.py` imports the Milestone F tripwire unchanged. §10 measured what it
catches: exactly one optimizer step lands, every step after it is silently skipped,
the loss stays finite and oscillating, and the progress bar keeps advancing. Here
that would be 12 hours and a saved adapter of pure garbage.

**Batch size is pinned to 1 with `--grad-accum 1`, deliberately.** The local 3060 row
that produced 47.2% ran at `effective_batch: 1`, so holding it fixed makes **the device
the only variable** — any accuracy difference is attributable to fp16 numerics and T4
throughput rather than to a different optimization. Changing two things at once is what
Milestone E's regression gate exists to prevent.

`--run-name qlora_t4` keeps this row separate from the committed 3060 row
(`results/train_qlora.json`, `outputs/adapters/qlora`). **Do not reuse the bare `qlora`
name** — a fresh clone would clobber the local row and then copy the wreckage out as an
artifact.

The adapter is saved every 200 steps, so a session that dies still leaves something
scoreable.

In [11]:
!python -m src.train \
    --budget-minutes 60 \
    --batch-size 1 \
    --grad-accum 1 \
    --lr 1e-4 \
    --save-every 200 \
    --run-name qlora_t4 \
    --out train_qlora_t4.json


--- QLoRA training — Week 3 -------------------------------------
Device  : Tesla T4
Budget  : 60 min wall-clock
Optim   : lr=0.0001 warmup=20 schedule=constant clip=None
Batch   : 1 physical x 1 accum = 1 effective
LoRA    : r=8 alpha=16 dropout=0.05

--- training pool (day-train — never the eval split) ------------
  1817 rows, 276 distinct scenes

--- model -------------------------------------------------------
  [VRAM] after 4-bit load             resident= 1.47 GiB   phase peak= 2.10 GiB
  [VRAM] after fp32 upcast            resident= 2.06 GiB   phase peak= 2.64 GiB
  fp32 upcast cost: +0.59 GiB resident
  trainable: 4,411,392 params (3,211,264 language + 1,200,128 vision)
  [VRAM] after LoRA attach            resident= 2.08 GiB   phase peak= 2.08 GiB

--- training ----------------------------------------------------
 step      loss        lr       |g|    |g|vis   |g|lang    scale  skip  ep  s/step
    0    2.0198  5.00e-06       nan       nan    10.494    65536  SKIP   0    1.0

## 8. Score it — benchmark row 2

`src/eval.py`, **unmodified**, with `--adapter`. A row scored by different code is not
a row.

**The reference is the night split, and the number to quote is the delta over the
per-question-type prior** — not the majority class. Answering `yes` to every yes/no
question and `car` to everything else needs no image, no training and no understanding,
because question type is readable straight off the question text
(`eval-protocol.md` Amendment 2):

| night, n=659 | zero-shot | local QLoRA (3060) |
|---|---|---|
| strict | 31.9% | **47.2%** |
| majority baseline (`always yes`) | 24.9% | 24.9% |
| **per-type prior** | **31.9%** | **31.9%** |
| **delta over the prior** | **+0.0 pp** | **+15.3 pp** |
| format compliance | 80.6% | 100.0% |

Zero-shot lands *exactly* on the prior — 210/659 either way. **This run should
reproduce the 47.2% accuracy**, since only the device changed. Its wall-clock and peak
VRAM, however, are the T4 numbers Week 4 inherits; the 3060's are not comparable.

~4 minutes for 659 rows.

In [12]:
!python -m src.eval \
    --split night \
    --adapter outputs/adapters/qlora_t4 \
    --run-name eval_qlora_t4 \
    --out eval_qlora_t4.json


Model   : nvidia/Cosmos-Reason2-2B  + adapter outputs/adapters/qlora_t4
Device  : Tesla T4
Split   : night  (659 rows, full_split=True)
Vocab   : 29 classes (from day-train)
Baseline: 24.9%  (always 'yes')

  adapter loaded: outputs/adapters/qlora_t4
  [VRAM] after load                   resident= 1.49 GiB   phase peak= 2.10 GiB
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  ... 0/659  strict=100.0%
  ... 50/659  strict=58.0%
  ... 100/659  strict=53.0%
  ... 150/659  strict=56.7%
  ... 200/659  strict=58.5%
  ... 250/659  strict=58.0%
  ... 300/659  strict=60.7%
  ... 350/659  strict=60.0%
  ... 400/659  strict=59.5%
  ... 450/659  strict=60.7%
  ... 500/659  strict=60.8%
  ... 550/659  strict=60.0%
  ... 600/659  strict=60.5%
  ... 650/659  strict=59.1%

strict exact-match :  58.9%   95% CI [55.1, 62.6]   <- reported
  cluster-adjusted :  58.9%   95% CI [55.0, 62.7]   (ICC=0.01

## 9. Is the difference real?

Paired McNemar on the same 659 examples. Overlapping confidence intervals are the
wrong test here — only the discordant pairs carry information about which method is
better, and the paired test is what makes n=659 enough to rank methods a couple of
points apart.

Both files must be scored on the **same split**; `--compare` now refuses outright if
they are not, rather than printing a warning above a p-value. The local run gave
b=98, c=199, **p = 6.5e-09**.

In [13]:
!python -m src.eval --compare results/eval_zeroshot_night.json results/eval_qlora_t4.json


A = eval_zeroshot_night.json   strict 31.9%
B = eval_qlora_t4.json   strict 58.9%
------------------------------------------------------------------------
  both correct          145
  A only (b)             65   <- discordant
  B only (c)            243   <- discordant
  neither               206

  McNemar p = 6.403e-24   [chi-square w/ continuity correction (b=65, c=243, chi2=101.718)]
  SIGNIFICANT at 0.05

Only the discordant pairs carry information about which method is better;
the concordant ones cancel. That is why this is more powerful than
comparing the two accuracies' confidence intervals.


## 10. Get the artifacts out

`/kaggle/working` is wiped on teardown beyond what the session saves. Copy the
adapter and the results files out **before** stopping the session, then commit the
JSON back to the repo (`results/` is tracked; adapters are not — they belong on the
Hub).

In [14]:
!mkdir -p /kaggle/working/artifacts
!cp -r outputs/adapters/qlora_t4 /kaggle/working/artifacts/
!cp results/train_qlora_t4.json results/eval_qlora_t4.json results/train_batch_probe_t4.json /kaggle/working/artifacts/
!cp outputs/adapters/qlora_t4/train_log.jsonl /kaggle/working/artifacts/
!du -sh /kaggle/working/artifacts/* && ls -la /kaggle/working/artifacts

192K	/kaggle/working/artifacts/eval_qlora_t4.json
20M	/kaggle/working/artifacts/qlora_t4
4.0K	/kaggle/working/artifacts/train_batch_probe_t4.json
2.2M	/kaggle/working/artifacts/train_log.jsonl
120K	/kaggle/working/artifacts/train_qlora_t4.json
total 2564
drwxr-xr-x 3 root root    4096 Aug 15 12:32 .
drwxr-xr-x 5 root root    4096 Aug 15 12:32 ..
-rw-r--r-- 1 root root  194935 Aug 15 12:32 eval_qlora_t4.json
drwxr-xr-x 2 root root    4096 Aug 15 12:32 qlora_t4
-rw-r--r-- 1 root root     834 Aug 15 12:32 train_batch_probe_t4.json
-rw-r--r-- 1 root root 2286330 Aug 15 12:32 train_log.jsonl
-rw-r--r-- 1 root root  118809 Aug 15 12:32 train_qlora_t4.json


## 11. Success criteria

- [ ] Every version printed in §3; `transformers` 4.5x, `huggingface_hub` <1.0 —
      **this is the one artifact that has never run; if anything breaks, it breaks here**
- [ ] `bf16_native = False` — still true, still the reason this is an fp16 run
- [ ] Leak check reports **0 shared images** (it is a hard failure, not a warning)
- [ ] Probe completes → **reconcile prediction #2** against the measured curve
- [ ] Training ends with `stop_reason = wall-clock budget exhausted`, **not** `tripwire`
- [ ] Scaler skips are a handful, not sustained (local: **7**, settling at scale **1024**)
- [ ] `eval_qlora_t4.json` strict accuracy lands near the local **47.2%**, and the
      quoted number is **`delta_over_prior_pp`** against the 31.9% prior — not the
      +22.3 pp the majority baseline would report
- [ ] McNemar runs against `eval_zeroshot_night.json` and does **not** refuse
- [ ] Artifacts copied out **before** stopping the session

**Then stop the session explicitly.** An idle session burns the ~30 hr/week quota.

In [1]:
import json
for f in ["train_batch_probe_t4.json", "train_qlora_t4.json", "eval_qlora_t4.json"]:
    d = json.load(open(f"results/{f}"))
    print("=" * 70); print(f)
    print(json.dumps({k: v for k, v in d.items()
                      if k not in ("records", "steps")}, indent=1))

FileNotFoundError: [Errno 2] No such file or directory: 'results/train_batch_probe_t4.json'